# jitmo tool-picker fine-tune (LoRA) and GGUF export

Trains a small model to pick the right tool among three candidates (docs/03-delivery/15, step 5).

**Before you run:** `Runtime > Change runtime type > GPU` (T4 is enough). Put `router_train.jsonl` and `router_train_args.jsonl` in your Drive folder `MyDrive/jitmo/`.

This notebook was written without being run. If a cell fails, send the error text to the coding agent.

**Never train on `testset/heldout.txt`.** It is not used here.

In [ ]:
#@title 1. Settings
BASE_MODEL = "unsloth/gemma-3-270m-it"  #@param ["unsloth/gemma-3-270m-it", "unsloth/functiongemma-270m-it", "Qwen/Qwen3.5-0.8B", "unsloth/gemma-3-1b-it"]
EPOCHS = 3  #@param {type:"integer"}
LEARNING_RATE = 2e-4  #@param {type:"number"}
LORA_RANK = 16  #@param {type:"integer"}
DATA = ["/content/drive/MyDrive/jitmo/router_train.jsonl", "/content/drive/MyDrive/jitmo/router_train_args.jsonl"]
OUT_DIR = "/content/drive/MyDrive/jitmo/out"
NAME = BASE_MODEL.split("/")[-1] + "-router"

In [ ]:
#@title 2. Install and mount Drive
!pip -q install -U transformers peft trl datasets accelerate sentencepiece gguf
from google.colab import drive
drive.mount('/content/drive')
import os; os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
#@title 3. Load the data (prompt/completion pairs; the loss is only on the answer)
import json, random
from datasets import Dataset
rows = [json.loads(l) for f in DATA for l in open(f, encoding='utf-8') if l.strip()]
random.Random(1).shuffle(rows)
data = [{"prompt": r["messages"][:2], "completion": r["messages"][2:]} for r in rows]
split = int(len(data) * 0.95)
train_ds, eval_ds = Dataset.from_list(data[:split]), Dataset.from_list(data[split:])
print(len(train_ds), 'train,', len(eval_ds), 'eval')

In [ ]:
#@title 4. Train
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float32)
lora = LoraConfig(r=LORA_RANK, lora_alpha=LORA_RANK*2, lora_dropout=0.05, task_type='CAUSAL_LM',
                  target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
cfg = SFTConfig(output_dir='/content/run', num_train_epochs=EPOCHS, learning_rate=LEARNING_RATE,
                per_device_train_batch_size=16, gradient_accumulation_steps=1, logging_steps=20,
                eval_strategy='epoch', save_strategy='no', report_to='none', max_length=512,
                completion_only_loss=True, lr_scheduler_type='cosine', warmup_ratio=0.05,
                bf16=torch.cuda.is_bf16_supported(), fp16=not torch.cuda.is_bf16_supported())
trainer = SFTTrainer(model=model, args=cfg, train_dataset=train_ds, eval_dataset=eval_ds,
                     processing_class=tok, peft_config=lora)
trainer.train()

In [ ]:
#@title 5. Quick check on the eval split (exact match of the answer)
model = trainer.model.eval()
ok = 0; n = min(150, len(eval_ds))
for ex in eval_ds.select(range(n)):
    text = tok.apply_chat_template(ex['prompt'], tokenize=False, add_generation_prompt=True)
    ids = tok(text, return_tensors='pt', add_special_tokens=False).to(model.device)
    out = model.generate(**ids, max_new_tokens=12, do_sample=False)
    got = tok.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    ok += got == ex['completion'][0]['content'].strip()
print(f'{ok}/{n} exact matches on held-back training data (the real test is jitmo-llmbench on testset/heldout.txt)')

In [ ]:
#@title 6. Merge the adapter and save the full model
merged = trainer.model.merge_and_unload()
merged.save_pretrained('/content/merged', safe_serialization=True)
tok.save_pretrained('/content/merged')

In [ ]:
#@title 7. Convert to GGUF and quantise to Q4_K_M (builds llama.cpp on the CPU of this machine, a few minutes)
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!pip -q install -r /content/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!python /content/llama.cpp/convert_hf_to_gguf.py /content/merged --outfile /content/{NAME}-f16.gguf --outtype f16
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null
!cmake --build /content/llama.cpp/build --target llama-quantize -j 4 > /dev/null
!/content/llama.cpp/build/bin/llama-quantize /content/{NAME}-f16.gguf /content/{NAME}-Q4_K_M.gguf Q4_K_M
!cp /content/{NAME}-Q4_K_M.gguf {OUT_DIR}/
!ls -l {OUT_DIR}

Done. Download `MyDrive/jitmo/out/<name>-Q4_K_M.gguf` and give the path to the coding agent. It is tested with `jitmo-llmbench` on `testset/heldout.txt`, which this notebook never saw.